# Stage 4, tier 3: full-transcript half-life and translation efficiency, behind their gate

Research plan **Priority 3** (full-transcript half-life) and **Priority 4** (translation
efficiency), both Gantt 42. The plan is explicit that these are gated, not assumed:

> **Feasibility gate.** Verify sequences include 5'/3' UTRs, CDS coordinates, labels, species
> and non-leaking train/test groups. If transcript regions are absent, do not treat it as a
> full-transcript test.

> **Gate.** Confirm the measured target (TE or mean ribosome load), full transcript / UTR /
> CDS availability, experimental context and valid grouped splits. Do not conflate TE with
> protein abundance.

So Part 1 is the gate, run on the published files themselves, and it decides what Parts 2 and
3 are allowed to claim. The gate is cheap: a few minutes and about 150 MB of downloads, no
GPU and no model. **Run Part 1 before committing a GPU night to Part 3.**

### Why this task exists at all

The stability benchmark BioLangFusion uses is coding sequence only. Stage 4 Phase A found
that its published split repeats training sequences in 68% of test rows, and that on the rows
it does not repeat, every model drops by 0.2 or more. A clean transcript-level half-life task
answers a question that benchmark cannot: **does the sequence outside the coding region carry
anything a codon-level model is missing**, and does fusion still help once it is there.

### The three questions, kept apart

The plan is emphatic that these labels are not interchangeable, and mislabelling them is the
most likely way to produce a wrong result that looks right:

| Target | What it measures | Dataset here |
| --- | --- | --- |
| RNA half-life | degradation rate of endogenous transcripts | `rnahl-human`, `rnahl-mouse` |
| Translation efficiency | ribosome profiling over transcript abundance | `translation-efficiency-human`, `-mouse` |
| Mean ribosome load | ribosomes per transcript isoform, **not** half-life | `mrl-sugimoto` |

`mrl-hl-lbkwk` carries one half-life target and one ribosome-load target on synthetic mRNAs,
and they are run separately. Nothing in this notebook pools two of these.

---

## Setup

In [ ]:
# Standard library
import json
import os
import time
from pathlib import Path

# Run any operation the Apple GPU backend lacks on the CPU; must precede importing torch
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Data handling
import numpy as np
import pandas as pd
import torch

# Shared project code in Code/shared_code; this notebook lives in Code/stage4
import sys
CODE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code").is_dir()), Path.cwd() / "Code")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code import analysis, encoders, mrnabench, sequences
from shared_code.encoders import ENCODERS

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| Apple GPU:", torch.backends.mps.is_available())

In [ ]:
RESULTS_DIR = Path("results") / "halflife_te"
EMBED_DIR = Path("../stage3/embeddings") / "mrnabench"
DATA_CACHE = mrnabench.DEFAULT_CACHE          # .data-cache/mrnabench, outside Git

# The two datasets Part 3 embeds. Human first; the mouse pair is the replication.
PRIMARY = ("rnahl-human", "translation-efficiency-human")
PROBE_SEEDS = (2541, 2547, 413, 412, 411, 321, 421, 2515, 2516)   # mRNABench's own nine
RUN_EMBEDDINGS = False                 # Part 3 only; read the cost projection first

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"device {DEVICE}; results in {RESULTS_DIR}, embeddings in {EMBED_DIR}")

## Part 1: the gate

Every number below is measured from the published file, not read from its description. The
coding region is not given as coordinates: each row carries a binary track marking the first
nucleotide of every codon, so the region is recovered from the marks and the spacing between
them is what proves the frame. A row whose marks are not three apart does not describe a
reading frame and is counted rather than quietly dropped.

The downloads are cached in `.data-cache/`, which Git ignores.

In [ ]:
gate_rows, frames = [], {}
for key, spec in mrnabench.DATASETS.items():
    started = time.perf_counter()
    frames[key] = mrnabench.load(key, DATA_CACHE)
    gate_rows.append(mrnabench.gate(frames[key], spec))
    print(f"  {key}: {len(frames[key]):,} rows in {time.perf_counter() - started:.0f} s")

gate = pd.DataFrame(gate_rows).set_index("dataset")
display(gate[["rows", "species", "targets present", "rows with a coding region",
              "rows with a 5' region", "rows with a 3' region", "median transcript nt",
              "longest transcript nt", "distinct genes"]]
        .style.format({"rows": "{:,}", "rows with a coding region": "{:.1%}",
                       "rows with a 5' region": "{:.1%}", "rows with a 3' region": "{:.1%}",
                       "median transcript nt": "{:,}", "longest transcript nt": "{:,}",
                       "distinct genes": "{:,}"}))

In [ ]:
for key, row in gate.iterrows():
    print(f"{key}\n    verdict: {row['verdict']}")
    if row["caveats"]:
        print(f"    watch:   {row['caveats']}")
    print(f"    measures: {row['measures']}")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
gate.to_csv(RESULTS_DIR / "gate.csv")

### What the gate decides

Read the verdicts above before going on. The rules this notebook then follows:

* A dataset whose rows carry both untranslated regions and a reading frame **may** be used
  for the region comparison of Part 2 and the fusion comparison of Part 3.
* A dataset without them is a sequence-level task and is reported as one. It does not answer
  Priority 3's question.
* Rows marking no coding region are **excluded from the region comparison and counted**, not
  silently dropped: on an isoform-resolved dataset they are the non-coding isoforms, and
  their share is itself a fact about the dataset.
* A dataset of a few hundred rows is kept out of the headline comparisons, because a grouped
  split of it leaves a test set too small to resolve the differences being compared.

In [ ]:
usable = [key for key, row in gate.iterrows() if row["verdict"].startswith("passes")
          and gate.loc[key, "rows"] >= mrnabench.MINIMUM_ROWS]
print("datasets that pass the gate and are large enough:")
for key in usable:
    print(f"  {key}: {mrnabench.DATASETS[key].measures}")
held_back = [k for k in mrnabench.DATASETS if k not in usable]
print("\nheld back from the headline comparison:")
for key in held_back:
    print(f"  {key}: {gate.loc[key, 'verdict']} | {gate.loc[key, 'caveats']}")

## Part 2: how much of the transcript matters, before any foundation model

The plan asks for "simple length and composition baselines" beside the encoders, and they
come first for a reason: if composition of the whole transcript already predicts half-life,
then an encoder has to beat that rather than beat nothing. Each region is scored on its own
with the same probe and the same grouped split, so the regions are compared on equal terms.

This part needs no GPU and no model: it is ridge on counted features.

In [ ]:
# The region sets the plan names. They are nested: "full transcript" carries the coding
# sequence's own features plus the two untranslated regions, so comparing the two rows asks
# whether the regions outside the coding sequence add anything.
REGION_SETS = {
    "CDS only": ("cds",),
    "CDS + 3' UTR": ("cds", "utr3"),
    "5' UTR only": ("utr5",),
    "3' UTR only": ("utr3",),
    "full transcript": ("utr5", "cds", "utr3"),
}
print("features per set, for one example row:")
example = mrnabench.region_frame(mrnabench.load(PRIMARY[0], DATA_CACHE).head(2))
for name, parts in REGION_SETS.items():
    print(f"  {name:<18} {mrnabench.region_features(example, parts).shape[1]} features")

In [ ]:
baseline_rows = []
for key in usable:
    spec = mrnabench.DATASETS[key]
    regions = mrnabench.region_frame(frames[key])
    keep = regions["frame_problem"] == ""
    regions = regions[keep].reset_index(drop=True)
    groups, grouping_described = mrnabench.grouping(regions, spec.species, DATA_CACHE)
    print(f"\n{key}: {len(regions):,} rows with a reading frame, grouped by {grouping_described}")
    for target in spec.targets:
        y = regions[target].to_numpy(dtype=float)
        for name, parts in REGION_SETS.items():
            X = mrnabench.region_features(regions, parts)
            scores = mrnabench.probe_over_seeds(X, y, groups, PROBE_SEEDS[:3])
            baseline_rows.append({"dataset": key, "target": target, "representation": name,
                                  "test r": scores["test_r"].mean(),
                                  "spread": scores["test_r"].std(),
                                  "seeds": len(scores)})
            print(f"    {name:<18} Pearson r {scores['test_r'].mean():.3f} "
                  f"(+/- {scores['test_r'].std():.3f} over {len(scores)} splits)")

baselines = pd.DataFrame(baseline_rows)
baselines.to_csv(RESULTS_DIR / "composition_baselines.csv", index=False)
display(baselines.pivot_table(index=["dataset", "target"], columns="representation",
                              values="test r").round(3))

## Part 3: the encoders, and whether fusion still helps on a whole transcript

The comparison the plan asks for, in its own words: 5' UTR, CDS, full transcript, full
transcript with a protein representation, and the three-encoder fusion, with the gain
reported **over the best full-transcript baseline** rather than over a weak regional input.

Two things make this more expensive than Phase A. The transcripts are long, a median of
about 3,000 nucleotides against the benchmarks' 999, and there are around 12,000 of them per
dataset. And every encoder except the k-mer ones has a context shorter than one transcript,
so a transcript is embedded in consecutive windows whose token states are concatenated before
pooling, which is mRNABench's own rule and is recorded as a deviation below.

The cell after next measures the cost before anything starts.

In [ ]:
# One encoder per modality, the same three BioLangFusion uses, so the comparison carries over.
STREAM_ENCODERS = {"dna": "ntv2_100m", "rna": "rnafm", "prot": "esm2_8m"}
INPUTS = {
    "5' UTR": ("rna", ("utr5",)),
    "CDS": ("rna", ("cds",)),
    "full transcript": ("rna", ("utr5", "cds", "utr3")),
    "full transcript, DNA encoder": ("dna", ("utr5", "cds", "utr3")),
    "protein of the CDS": ("prot", ("cds",)),
}

for key in PRIMARY:
    if key not in usable:
        print(f"{key} did not pass the gate; Part 3 skips it.")
print(f"\nwindows per encoder: " + ", ".join(
    f"{k}: {mrnabench.default_window(ENCODERS[v]):,} characters"
    for k, v in STREAM_ENCODERS.items()))

In [ ]:
def stream_texts(regions, modality, parts):
    # The strings one encoder reads for a region set: nucleotides in its own alphabet, or the
    # translation of the coding sequence for a protein encoder.
    encoder = ENCODERS[STREAM_ENCODERS[modality]]
    if modality == "prot":
        return [mrnabench.encoder_text(encoder, row["cds"]) for _, row in regions.iterrows()]
    joined = ["".join(row[p] for p in parts) for _, row in regions.iterrows()]
    return [mrnabench.encoder_text(encoder, text) for text in joined]


def cost_projection(regions, seconds_per_1000_nt=0.02):
    # A rough projection from total nucleotides, refined by the timing cell below.
    total = sum(len(row["utr5"]) + len(row["cds"]) + len(row["utr3"])
                for _, row in regions.iterrows())
    return total / 1000 * seconds_per_1000_nt / 3600

In [ ]:
if RUN_EMBEDDINGS:
    embedded = {}
    for key in PRIMARY:
        spec = mrnabench.DATASETS[key]
        regions = mrnabench.region_frame(frames[key])
        regions = regions[regions["frame_problem"] == ""].reset_index(drop=True)
        for name, (modality, parts) in INPUTS.items():
            texts = stream_texts(regions, modality, parts)
            folder = EMBED_DIR / key
            tag = name.replace("'", "").replace(" ", "_").replace(",", "")
            started = time.perf_counter()
            matrix, reused = mrnabench.embed_with_cache(
                STREAM_ENCODERS[modality], texts, folder / f"{tag}.npz", device=DEVICE)
            embedded[(key, name)] = matrix
            print(f"  {key} / {name}: {matrix.shape}, "
                  f"{'reused' if reused else f'{(time.perf_counter() - started) / 60:.1f} min'}")
else:
    print("RUN_EMBEDDINGS is off. Part 1 and Part 2 run without it; turn it on for a GPU night.")

In [ ]:
if RUN_EMBEDDINGS and embedded:
    rows_out = []
    for key in PRIMARY:
        spec = mrnabench.DATASETS[key]
        regions = mrnabench.region_frame(frames[key])
        regions = regions[regions["frame_problem"] == ""].reset_index(drop=True)
        groups, described = mrnabench.grouping(regions, spec.species, DATA_CACHE)
        for target in spec.targets:
            y = regions[target].to_numpy(dtype=float)
            available = {name: embedded[(key, name)] for _, name in
                         [(k, n) for (k, n) in embedded if k == key]}
            available["full transcript + protein"] = np.hstack(
                [available["full transcript"], available["protein of the CDS"]])
            available["three encoders fused"] = np.hstack(
                [available["full transcript, DNA encoder"], available["full transcript"],
                 available["protein of the CDS"]])
            for name, X in available.items():
                scores = mrnabench.probe_over_seeds(X, y, groups, PROBE_SEEDS)
                rows_out.append({"dataset": key, "target": target, "representation": name,
                                 "test r": scores["test_r"].mean(),
                                 "standard error": scores["test_r"].std() / np.sqrt(len(scores)),
                                 "test Spearman": scores["test_spearman"].mean(),
                                 "seeds": len(scores), "grouping": described})
                print(f"  {key} / {name}: r {scores['test_r'].mean():.3f}")
    encoder_scores = pd.DataFrame(rows_out)
    encoder_scores.to_csv(RESULTS_DIR / "encoder_probes.csv", index=False)
    display(encoder_scores.pivot_table(index=["dataset", "target"], columns="representation",
                                       values="test r").round(3))
else:
    print("Nothing to score yet.")

In [ ]:
# The plan's definition, applied: the gain over the best FULL-TRANSCRIPT baseline, not over a
# weak regional input. Both the best single encoder and the composition floor are included,
# because a fusion that beats one and not the other has not earned the claim.
if RUN_EMBEDDINGS and "encoder_scores" in dir() and len(encoder_scores):
    gains = []
    for (key, target), block in encoder_scores.groupby(["dataset", "target"]):
        by_name = block.set_index("representation")["test r"]
        singles = [n for n in ("full transcript", "full transcript, DNA encoder",
                               "protein of the CDS") if n in by_name]
        best_single = by_name[singles].max()
        floor = baselines.query("dataset == @key and target == @target and "
                                "representation == 'full transcript'")["test r"]
        for fused in ("full transcript + protein", "three encoders fused"):
            if fused in by_name:
                gains.append({
                    "dataset": key, "target": target, "fused model": fused,
                    "fused r": by_name[fused], "best single r": best_single,
                    "delta fusion": by_name[fused] - best_single,
                    "over composition": by_name[fused] - (floor.iloc[0] if len(floor) else np.nan),
                })
    display(pd.DataFrame(gains).round(3))
    pd.DataFrame(gains).to_csv(RESULTS_DIR / "fusion_gains.csv", index=False)

## What this notebook may and may not conclude

* A positive gain here is **incremental predictive benefit under this protocol**, as the plan
  says, not evidence of independent biological information. The probe, the split and the
  pooling are all fixed across rows of the table, and that is the whole of the claim.
* The absolute values are not comparable with mRNABench's published table: that benchmark
  reports its own models under its own homology splits, and uses chunked embeddings for
  models whose context is short, which this notebook also does but with different encoders.
* Half-life, translation efficiency and mean ribosome load are three different measurements.
  Every table above is keyed by dataset and target so that none of them can be read as
  another.

## Deviations recorded for this notebook

| # | deviation | why |
| --- | --- | --- |
| 1 | **Grouping by gene rather than by the benchmark's homology groups**, unless the homology map has been downloaded into `.data-cache/mrnabench/homology_maps/` | the map is a separate download; grouping by gene is strictly weaker, because paralogues can still sit on both sides, so this is recorded rather than passed off as the benchmark's own split |
| 2 | A transcript longer than an encoder's context is embedded in consecutive windows, and the token states are concatenated before pooling | mRNABench's own rule for short-context models; a window cannot see outside itself, so a chunked embedding is an approximation of the whole transcript rather than the thing itself |
| 3 | Frozen encoders with a ridge probe, where RiboNN and Saluki train end to end | the project's premise is frozen fusion, and the plan asks for the gain over the best full-transcript baseline under one protocol, not for a new state of the art |
| 4 | Nine split seeds in Part 3, three in Part 2 | Part 2 compares five region sets across several datasets and targets, where three splits already separate them; Part 3 reports the differences the project will quote, and uses the benchmark's own nine |
| 5 | Rows marking no coding region are excluded from the region comparison | they have no CDS to compare against, and their share is reported in the gate instead |
| 6 | ESM-2 8M as the protein encoder, not a larger one | it is BioLangFusion's own protein encoder, so a gain here is comparable with Phase A's |